# AST 6215 Galactic Astronomy Project 1: Stellar Modeling
The goal of this project is for you to experience a bit more of how stellar modeling is done. We're going to using the open source stellar evolution code YREC for this (https://yrec.readthedocs.io/en/latest/). Specifically, we'll be adapting some of the exercises developed for the MESA Summer School @ Konkoly Observatory 2023 (https://mesahub.github.io/summer-school-2023/agenda/) by Giulia Cinquegrana, Sunny Wong, Attila Bódi, and Jamie Tayar.



## Task 1: Run a Model
Stellar evolution codes are set up such that you define the set of initial parameters (.nml1 file, https://yrec.readthedocs.io/en/latest/namelist_control.html), tell it what physical assumptions to use (.nml2 file, https://yrec.readthedocs.io/en/latest/namelist_physics.html), and then it will evolve the model in time as requested. 



Normally to run a stellar evolution calculation, you need to install the evolution code on your own computer. This works great 85% of the time, and is horrible to debug the other 15 percent of the time. For this exercise, you have a choice, you can either install the YREC stellar evolution code (https://github.com/yreclab/yrec) or you can use the web version (https://yreclab.github.io/). For your run, use the Test_m1p0feh+0p0GS98_base_TAMS Test Suite Case (testsuite/evolve_to_tams then Test_m1p0feh+0p0GS98_base_TAMS). The only change you need to make is to NPRTMOD on line 61 of the .nml1 file, please change that from 1000 to 300. 

**Question:** 
What is the default mass and metallicity assumed for this model?


**Answer:**

Generally these models run until you tell them to stop. YREC allows you to add stopping conditions of various types in the .nml1 file. 

__Question:__ What is the stopping condition that is not commented out (preceeded by a !)? What does that mean physically?

__Answer:__

In a few minutes you should have a model. You'll want to download the .track and .store files, and if they aren't called things like Test_m1p0feh+0p0GS98_base_TAMS.track, consider renaming them

Open the file called Test_m1p0feh+0p0GS98_base_TAMS.track. Find the column labeled Mass_msun. 

**Question:** What is the first entry in star_mass? How does that agree with the mass you asked it to run? 

**Answer:**

## Task 2: Read the model into python
You are welcome to do this any way that works for you (astropy tables, pandas, etc). I tend toward astropy tables


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, join, unique

In [ ]:
#to read in the file (you will likely have to alter the name for your own file)
star1= Table.read('Test_m1p0feh+0p0GS98_base_TAMS.track', format='ascii')

Use the following code to print out the columns available in your file

In [ ]:
print(star1.columns)

**Question:** Print the mass of the first model- does it agree with what you found in the file itself?

In [ ]:
print(star1['Mass_msun'][0]) # model "star1" column "star_mass" item [0] in the array

**Answer:**

**Question:** Find the central metallicity of the first step (Z_cen) Does it agree with the number you requested?

**Answer:**

## Task 3: Plot the model
Some people can look at a list of numbers and understand what is happening. I cannot, I need to see a plot. Create a plot of log Luminosity versus surface temperature for your model. You'll want to be careful of which measurments are in log and which are not. 

In [ ]:
plt.plot(10**star1['log_Teff'], star1['LogL_lsun'])
plt.ylabel(r'Log(L [L$_{\odot}$] )', fontsize=15)  # ---- CHANGE Y AXIS LABEL HERE ---- #
plt.xlabel(r'(T$_{\rm eff}$ [K])', fontsize=15)  # ---- CHANGE X AXIS LABEL HERE ---- #
plt.gca().invert_xaxis() # ---- REVERSES X-AXIS: TURN OFF IF NOT PLOTTING HRD ---- #
#plt.savefig('my_figure.eps', bbox_inches='tight', format='eps')

**Question:** What parts of the stellar evolution have you plotted?

**Answer:**

**Question:** How old is the model when it stops running? Why did it stop?

In [ ]:
#code here to get age of the last model, etc

**Answer:**

Zoom in on your plot around the parameters of the sun, and mark the the IAU solar values T=5772 K, Log(L)=0. 

In [ ]:
# code to make plot
# you might find it helpful to use something like plt.gca().set_xlim([5800, 5700])


**Question:** Does your model go through the solar calibrated value? Describe any offset.

**Answer:**

**Question:** Find the model point closest to the age of the sun. Add a point marking that model on your plot.

In [ ]:
# I used something like np.where() for this, but there are also better ways to do this

**Question:** Given what we discussed in class, or other things you know, why might a solar model not go through the solar values? How is a model forced to go through the solar values? 

**Answer:**

## Task 4: What does the profile look like?

You'll see lots of plots of stellar evolution tracks, but having the models also allows you to see what's really going on inside. To do that, you need to load in the profile outputs, which we changed it to put out every 300 steps. 

We are going to pick a profile close to the solar age, 4.5 Gyrs for this one solar mass solar metallicity model. 

In [ ]:
profilelist1=Table.read('Test_m1p0feh+0p0GS98_base_TAMS.store', format='ascii')
profilelist1.add_column(profilelist1['MODEL'], name='Step')

Let's figure out what steps on the track have profiles, match those to the track file, and identify what the Age, surface temperature, and total Luminosity of the star are at steps where we saved the profile.

In [ ]:
unique_profileslist1 =unique(profilelist1, keys='MODEL')
unique_profileslist1Table=Table(unique_profileslist1['MODEL','Step'])
trackhasprofile_1=join(unique_profileslist1, star1, join_type='inner')
print(trackhasprofile_1['MODEL', 'Age_gyr', 'LogL_lsun', 'log_Teff'])

We want to select a profile to look at that is part way through the main sequence, but not quite at the turnoff yet

In [ ]:
ProfileNumber=600  #change this number for your model?
profile1_mainsequence=profilelist1[np.where(profilelist1['MODEL']==ProfileNumber)] 

In [ ]:
#print the values available for each shell here
print(profile1_mainsequence.columns)

**Question:** In which order are the data stored in this file (is it center at the top, surface at the bottom, or vice versa?)

In [ ]:
#some print statements here maybe?

**Answer:** 

**Question:** Plot the nuclear burning luminosity (E_NUC) and the opacity (OPACITY) on the same plot, as a function of the mass coordinate. Is the nuclear burning region in this model convective? 

In [ ]:
#One way to do that
Msun=1.99e33
convective_region=np.where(profile1_mainsequence['CONVECTIVE']=='T')
ax1 = plt.subplot() 
plt.plot(10**profile1_mainsequence['MASS']/Msun, profile1_mainsequence['E_NUC'])
plt.plot(10**profile1_mainsequence['MASS']/Msun, profile1_mainsequence['OPACITY'])
plt.scatter(10**profile1_mainsequence['MASS'][convective_region]/Msun, profile1_mainsequence['MASS'][convective_region]*0+15, color='pink')
plt.gca().set_ylim([0, 20])
plt.xlabel(r'Mass M$_{\odot}$', fontsize=15)
plt.ylabel(r'$\epsilon_{nuc}$', fontsize=15, color='b') 
ax2 = ax1.twinx()
plt.ylabel(r'Opacity', fontsize=15, color='orange') 
plt.savefig('SolarModelMainSequence_nuc_convec_solar.png', bbox_inches='tight', format='png')

**Answer:**

**Question:** By plotting the gradients (adiabatic gradient, DELAD; radiative gradient, DELR; temperature gradient, DEL), explain why different parts of the star are radiative or convective. How does this compare to what we discussed in class?

In [ ]:
#code to make plot here

**Answer:** 

## Task 5: What are the effects of changing the stellar mass?

Now you are going to run models at different masses, specifically 1.3 and 1.6 Msun. To do this, you'll want to make a copy of the files (I called mine Test_m1p3feh+0p0GS98_base_TAMS),  rescale the mass of the star (RSCLM(1)=1.3 on line 27 of .nml1) and change the name of the output files in nml1 (I called mine */TEST_m1p3feh+0GS98_base_TAMS *, you can do a find-replace or something more sophisticated). You may also need to start what conditions its starting from (FFIRST on line 95 - m1600gs98z018804_Dbl.first worked for me for the 1.6 Msun star). 


**Question:** Plot all three models on the same graph of Log(Luminosity) vs. Effective temperature. How does the temperature and luminosity of the star react to a change in the stellar mass?

In [ ]:
#code to read in models

In [ ]:
#code to plot the graphs

**Answer:**

**Question:** How do the main sequence lifetimes of these stars change as you change the stellar mass? How does this compare to what you calculated in Problem Set 1?

In [ ]:
#code to print or plot things

**Answer:** 

**Question:** Look at the structure of these models part way through the main sequence. How do their structures differ? Can you explain this using the nuclear luminosity, gradients, opacity, etc?

In [ ]:
#code to read in profiles

In [ ]:
#code to make graphs

In [ ]:
#more code to make graphs

**Answer:**

## Task 6: What are the effects of changing the stellar metallicity?
Now we want to test the effects of changing the metallicity. Go back to the 1 M$_\odot$ model. Make a copy e.g. Test_m1p0feh-1p0GS98_base_TAMS and decrease the Z by a factor of 10. The main control for this is  RSCLZ(1) on line 30. However, there are several other places where the abundance is used for something (labels for the atmosphere, telling it which equation of state table to use, etc). So I did a find replace on .016 and made it .0016. This worked everywhere except the equation of state table, line 86, where I had to change FOPALE06 to end in EOSOPAL06Z0.001646600. Don't forget to also change the name of your output files to avoid overwritting your previous work. 

For those of you paying a lot of attention, you'll note that Fe/H is not quite the same thing as Z, and we haven't altered Y to really make things match. Ignore that for now. 

**Question:** Make a plot to show how the change in metallicity is changing the main sequence Luminosity and Temperature. Estimate what mass model at solar metallicity would be most similar in **main sequence** temperature to this 1 solar mass, one tenth solar metallicity model. (I encourage you to estimate a value between the values of the models you have run.)

In [ ]:
#code to read in model

In [ ]:
#code to make plot

**Answer:**

**Question:** How does changing the metallicity affect the opacity of the model?

In [ ]:
#code to read in profile

In [ ]:
#code to make plot

**Answer:**

**Question:** How does changing the metallicity affect the energy generation rate (nuclear luminosity)? How does this relate to position on the HR diagram?

In [ ]:
#code to make plot

**Answer:**

## Task 7: Explore the physics
So far, we have been using the default values for all of the physical assumptions used in making the model (some of the things in .nml1 and everything in .nml2). Pick one of these assumptions to explore further. I encourage you to pick something different from what other people are choosing.

**Question:** Through some combination of looking through your notes, googling it, reading the YREC documention (https://yrec.readthedocs.io/en/latest/namelist_physics.html, https://yrec.readthedocs.io/en/latest/namelist_control.html) or skimming relevant papers (https://ui.adsabs.harvard.edu/abs/2026ApJ..1006..220P/abstract), etc. explain what this control is related to physically in a few sentences. 

**Answer:**

Using the same mass and metallicity of one of your previous models, choose a different value for this physical assumption, and run the model.

**Question:** What parameter did you change, and what value did you change it to?

**Answer:**

**Question:** How did your change in the physical assumption change the resulting model?

In [ ]:
#code to read in track
#code to read in profile

In [ ]:
#code to make plots

**Answer:** 

**Question:** Using what you have learned in class or in your reading, can you explain why the physical choice you made mattered in the observed way?

**Answer:** 

**Question:** Think back to the comparison of the 1 solar mass model to the observed solar properties. Do you expect that some change in your parameter could help bring a solar model into better agreement? Why or why not? (You don't actually have to run the model for this question, just think about what is happening, where, and why)

**Answer:**

## Task 8: Write it Up
Great, you have done a science experiment. Now you will practice writing about it. Specifically, I want you to write the abstract for the hypothetical paper you could write about your experiement in Task 7. It should:
* read like a formal paper abstract (structure, langauage, etc)
* have approximately 5 sentences
* follow the hourglass structure we discussed in class
* comply with the American Astronomical Society Journal guidelines (https://journals.aas.org/manuscript-preparation/) Specifically: The abstract should summarize concisely the content and conclusions of the article. The abstract should be a single paragraph of not more than 250 words.


**Question:** Who is the audience for this hypothetical paper (please be more specific than "astronomers")

**Answer:**

**Question:** Why would they be interested in this paper? What would they be looking to take away from it? (this should feed into your introductory sentences in the hourglass, as well as what conclusions you include, and how you connect them to the bigger picture at the end)

**Answer:**

**Question:** Write your abstract here.

**Answer:**